In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
import pickle
import os

In [ ]:
ref_seq_table = "/p/lustre1/golez1/vog_231_ml_training_table_random_sample/training.tsv"

In [ ]:
ORDERED_TAXA_RANKS = ["Domain", "Realm", "Kingdom", "Phylum", "Subphylum", "Class", "Order", "Suborder", "Family", "Subfamily", "Genus", "Species"]

In [ ]:
header = pd.read_csv(ref_seq_table, sep="\t", nrows=0)
col_names = header.columns.tolist()

dtype_dict = {col_names[0]: str}
for col in col_names[1:-len(ORDERED_TAXA_RANKS)]:
    dtype_dict[col] = float
for col in col_names[-len(ORDERED_TAXA_RANKS):]:
    dtype_dict[col] = str

In [ ]:
data_df = pd.read_csv(ref_seq_table, sep="\t", nrows=10000, dtype=dtype_dict)

x_columns = [col for col in data_df.columns if col not in ["gene"] + ORDERED_TAXA_RANKS]
y_columns = ORDERED_TAXA_RANKS[1:]  # remove Domain and Species

X = data_df[x_columns].to_numpy()
y = data_df[y_columns].fillna("").to_numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
from sklearn.preprocessing import LabelEncoder

y_train_encoded = np.zeros((y_train.shape[0], y_train.shape[1]), dtype=int)
mappings = dict()

for i in range(len(y_columns)):
    le = LabelEncoder()
    y_train_encoded[:, i] = le.fit_transform(y_train[:, i])
    mappings[i] = dict(enumerate(le.classes_))

with open(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "encoding_map.tsv"), "w") as file:
    file.write("Column\tEncoding\tDecoding\n")
    for i in mappings.keys():
        for encoding, decoding in mappings[i].items():
            file.write(f"{i}\t{encoding}\t{decoding}\n")

In [ ]:
def test_model_accuracy(clf, mappings, X_test, y_test):
    y_pred = clf.predict(X_test)

    max_len = 0
    for i in mappings.keys():
        for decoding in mappings[i].values():
            if len(decoding) > max_len:
                max_len = len(decoding)

    y_pred_decoded = np.zeros((y_pred.shape[0], y_pred.shape[1]), dtype=f'<U{max_len}')
    for i in range(y_pred.shape[1]):
        for j, encoding in enumerate(y_pred[:, i]):
            y_pred_decoded[j, i] = mappings[i][encoding]

    # print(y_pred_decoded[0, :])

    for i in range(len(y_columns)):
        test_col_arr = y_test[:, i]
        pred_col_arr = y_pred_decoded[:, i]

        # print(f"{round(len([name for name in test_col_arr if name not in  mappings[i].values()]) / len(test_col_arr) * 100, 2)} % not in possible classifications")
        
        print(f"Column: {y_columns[i]}")
        print(f"Accuracy: {round(accuracy_score(test_col_arr, pred_col_arr), 2)}")
        # print(classification_report(test_col_arr, pred_col_arr, zero_division=0))
        print()

In [ ]:
def save_model(filepath, model):
    with open(filepath, 'wb') as file:
        pickle.dump(model, file)

def load_model(filepath):
    with open(filepath, 'rb') as file:
        return pickle.load(file)

XGBoost

In [ ]:
import xgboost as xgb

xgb_model = MultiOutputClassifier(
    xgb.XGBClassifier(
        objective='binary:logistic',
        max_depth=3,
        learning_rate=0.1,
        n_estimators=50,
        enable_categorical=True
    )
).fit(X_train, y_train_encoded)

In [ ]:
save_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "xgb_model.pkl"), xgb_model)

In [ ]:
xgb_model = load_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "xgb_model.pkl"))

In [ ]:
test_model_accuracy(xgb_model, mappings, X_test, y_test)

Neural Network

In [ ]:
from sklearn.neural_network import MLPClassifier

clf = MultiOutputClassifier(
    MLPClassifier(
        random_state=1, 
        max_iter=300
    )
).fit(X_train, y_train_encoded)

In [ ]:
save_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "neural_network.pkl"), clf)

In [ ]:
clf = load_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "neural_network.pkl"))

In [ ]:
test_model_accuracy(clf, mappings, X_test, y_test)

Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = MultiOutputClassifier(LogisticRegression(solver="saga", max_iter=1000)).fit(X_train, y_train)

In [ ]:
save_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "logistic_regression.pkl"), lr)

In [ ]:
test_model_accuracy(lr, X_test, y_test)

KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=3)).fit(X_train, y_train)

In [ ]:
save_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "knn.pkl"), knn)

In [ ]:
test_model_accuracy(knn, X_test, y_test)

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)).fit(X_train, y_train)

In [ ]:
save_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "random_forest.pkl"), forest)

In [ ]:
test_model_accuracy(forest, X_test, y_test)

Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb = MultiOutputClassifier(GaussianNB()).fit(X_train, y_train)

In [ ]:
save_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "naive_bayes.pkl"), nb)

In [ ]:
test_model_accuracy(nb, X_test, y_test)

Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = MultiOutputClassifier(GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=42)).fit(X_train, y_train)

In [ ]:
save_model(os.path.join("/p/lustre1/golez1/vog_231_ml_training_table_random_sample", "gradient_boosting.pkl"), gb)

In [ ]:
test_model_accuracy(gb, X_test, y_test)